# Lesson 17: Deployment, serving and LLMOps

*Worked solutions for [the lesson page](../../lessons/17-deployment-llmops.html). The exercise version without solutions is `../17-deployment-llmops.ipynb`.*

Take an AI system from notebook to production by building a streaming API service, serving open models efficiently with quantisation and the right memory budget, controlling latency and cost, and making everything observable.

**Runs on:** Any computer (CPU). Running the service needs an API key and a terminal.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q numpy fastapi "uvicorn[standard]" anthropic httpx

## A streaming API with FastAPI

FastAPI is the common choice for Python AI services: typed request validation with Pydantic, async support for many concurrent slow model calls, and automatic API docs.

Run in a terminal:

```bash
uv add fastapi "uvicorn[standard]" anthropic pydantic
```

In [ ]:
%%writefile app.py
import json, logging, time, uuid
import anthropic
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field

MODEL = "claude-sonnet-5-5"
log = logging.getLogger("app")
logging.basicConfig(level=logging.INFO, format="%(message)s")

app = FastAPI(title="Instrument assistant")
client = anthropic.AsyncAnthropic(max_retries=3, timeout=60)

class AskRequest(BaseModel):
    question: str = Field(min_length=1, max_length=4000)

@app.get("/health")
async def health():
    return {"status": "ok"}

@app.post("/ask")
async def ask(req: AskRequest):
    t0 = time.perf_counter()
    r = await client.messages.create(model=MODEL, max_tokens=800,
                                     messages=[{"role": "user", "content": req.question}])
    log.info(json.dumps({"event": "ask", "latency_s": round(time.perf_counter() - t0, 3),
                         "in": r.usage.input_tokens, "out": r.usage.output_tokens}))
    return {"answer": r.content[0].text}

@app.post("/ask/stream")
async def ask_stream(req: AskRequest, request: Request):
    request_id = str(uuid.uuid4())

    async def events():
        t0, first = time.perf_counter(), None
        try:
            async with client.messages.stream(model=MODEL, max_tokens=800,
                                              messages=[{"role": "user", "content": req.question}]) as stream:
                async for text in stream.text_stream:
                    if await request.is_disconnected():          # user left: stop generating and paying
                        return
                    first = first or time.perf_counter()
                    yield f"data: {json.dumps({'text': text})}\n\n"   # server-sent events format
                final = await stream.get_final_message()
            yield f"data: {json.dumps({'done': True})}\n\n"
            log.info(json.dumps({"event": "ask_stream", "id": request_id,
                                 "ttft_s": round((first or time.perf_counter()) - t0, 3),
                                 "total_s": round(time.perf_counter() - t0, 3),
                                 "in": final.usage.input_tokens, "out": final.usage.output_tokens}))
        except anthropic.APIError as e:
            log.error(json.dumps({"event": "upstream_error", "id": request_id, "error": repr(e)}))
            yield f"data: {json.dumps({'error': 'The model service is busy. Please try again.'})}\n\n"

    return StreamingResponse(events(), media_type="text/event-stream")

Run in a terminal:

```bash
uv run uvicorn app:app --reload
curl -N -X POST localhost:8000/ask/stream -H "Content-Type: application/json" \
     -d '{"question": "Explain the KV cache in three sentences."}'
```

### Ship it in a container

*Dockerfile*

```dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY pyproject.toml uv.lock ./
RUN pip install uv && uv sync --frozen --no-dev
COPY . .                       # list .venv and .env in .dockerignore so they are not copied
EXPOSE 8000
CMD ["uv", "run", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]
```

## Serving open models

### The serving landscape

Run in a terminal:

```bash
# vLLM: an OpenAI-compatible server on port 8000
pip install vllm
vllm serve Qwen/Qwen2.5-7B-Instruct --max-model-len 16384 --gpu-memory-utilization 0.9

# serve a base model plus your LoRA adapter from lesson 16
vllm serve Qwen/Qwen2.5-0.5B-Instruct --enable-lora --lora-modules notes=./notes-to-json-lora
```

### The GPU memory budget

In [ ]:
def serving_memory_gb(params_billion, weight_bytes, layers, kv_heads, head_dim,
                      context_tokens, concurrent_seqs, kv_bytes=2):
    weights = params_billion * 1e9 * weight_bytes
    kv_per_token = 2 * layers * kv_heads * head_dim * kv_bytes      # keys and values, every layer
    kv_total = kv_per_token * context_tokens * concurrent_seqs
    return round(weights / 1e9, 1), round(kv_total / 1e9, 1)

# an 8B model with 32 layers, 8 KV heads of dimension 128:
print(serving_memory_gb(8, 2, 32, 8, 128, 8192, 16))     # (16.0, 17.2): BF16 weights + 16 users at 8k context
print(serving_memory_gb(8, 0.5, 32, 8, 128, 8192, 16))   # (4.0, 17.2): 4-bit weights; KV cache now dominates

## Latency, throughput and cost

### Levers, roughly in order of payoff

In [ ]:
import numpy as np

class SemanticCache:
    def __init__(self, embed, threshold=0.95):
        self.embed, self.threshold = embed, threshold
        self.vecs, self.answers = [], []

    def get(self, question: str):
        if not self.vecs:
            return None
        q = self.embed(question)
        sims = np.array(self.vecs) @ q
        i = int(sims.argmax())
        return self.answers[i] if sims[i] >= self.threshold else None

    def put(self, question: str, answer: str):
        self.vecs.append(self.embed(question)); self.answers.append(answer)

In [ ]:
import re, hashlib

def toy_embed(text, dim=256):
    """Hashed bag of words: a stand-in for a real embedding model, good enough to show the mechanics."""
    v = np.zeros(dim)
    for w in re.findall(r"[a-z0-9]+", text.lower()):
        v[int(hashlib.md5(w.encode()).hexdigest(), 16) % dim] += 1
    return v / (np.linalg.norm(v) or 1)

cache = SemanticCache(toy_embed, threshold=0.8)
cache.put("How do I replace the desiccant?", "Open the enclosure, swap the cartridge, check the indicator is blue.")
for q in ["how do I replace the desiccant", "How do I replace the filter wheel belt?"]:
    print(f"{q!r:<45} -> {cache.get(q)}")

## Observability

### Tracing

Tools: OpenTelemetry with its generative-AI conventions for vendor-neutral tracing; LLM-specific platforms such as Langfuse (open source), Arize Phoenix, LangSmith, Helicone or Weights & Biases Weave, which add prompt management, cost tracking, datasets and evals on top of traces.

In [ ]:
# a minimal hand-rolled span recorder; production code would use OpenTelemetry or a tracing SDK
import contextlib, json, time, uuid

@contextlib.contextmanager
def span(name, trace_id, **attrs):
    t0 = time.perf_counter()
    record = {"trace": trace_id, "span": name, **attrs}
    try:
        yield record                      # callers add outputs, token counts, etc.
    except Exception as e:
        record["error"] = repr(e); raise
    finally:
        record["ms"] = round(1000 * (time.perf_counter() - t0), 1)
        print(json.dumps(record))         # ship to your log pipeline

trace_id = str(uuid.uuid4())
with span("retrieve", trace_id, query="belt slipping") as s:
    s["chunk_ids"] = ["manual-x#12", "manual-x#13"]

> **Traces contain user data**
>
> Prompts and outputs often include personal or confidential information. Decide what to store, redact where needed, restrict access, and set retention periods. Privacy obligations apply to logs too.

### Hands-on: production-ise your RAG system

1. Wrap your lesson 13 `MiniRAG` in a FastAPI service with `/ask` and `/ask/stream`, validating input with Pydantic.
2. Log one JSON line per request with: request id, retrieved chunk ids, model, input and output tokens, TTFT, total latency and estimated cost.
3. Add an exact-match response cache, then a semantic cache, and measure hit rates on a replay of 100 questions (include paraphrases).
4. Write a load test (for example with `asyncio` firing 50 concurrent requests, or the Locust tool) and report p50 and p95 latency.
5. Containerise it and run it with `docker run`.

#### Solution: what to measure and typical findings

In [ ]:
# Needs a running server (see the text above)
import asyncio, time, httpx, numpy as np

async def one(client, q):
    t0 = time.perf_counter()
    r = await client.post("http://localhost:8000/ask", json={"question": q}, timeout=120)
    return time.perf_counter() - t0, r.status_code

async def load_test(questions, concurrency=50):
    async with httpx.AsyncClient() as client:
        res = await asyncio.gather(*(one(client, q) for q in questions[:concurrency]))
    lat = np.array([t for t, code in res if code == 200])
    print(f"ok {len(lat)}/{len(res)}  p50 {np.percentile(lat, 50):.2f}s  p95 {np.percentile(lat, 95):.2f}s")

# asyncio.run(load_test(my_questions))

Typical findings: p95 is often two to three times p50, driven by long answers and provider-side queueing; a few requests may hit rate limits under burst load, which your retries should absorb (check the logs). The semantic cache catches paraphrases that the exact cache misses, but with a loose threshold it also returns answers to subtly different questions; reading those false hits is the best way to choose the threshold. The JSON logs make all of this answerable in a few lines of pandas (lesson 01 skills, paying off).

## Check yourself

Take the quiz at the end of [the lesson page](../../lessons/17-deployment-llmops.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.